# Insurance (Claims Operations) — Pre-Filled Configuration
### Ready-to-Run Config for the Insurance Use Case

This is a **pre-configured** version of `00_Industry_Config` specifically for the
**Claims Adjuster Documentation Burden** demo. All table lists, artifact names,
and data paths are hardcoded from the existing 23-CSV insurance dataset.

**Use this instead of `00_Industry_Config` if you want to skip auto-discovery and run
directly with the known insurance tables.**

---

### Data Summary
- **6 Dimensions** — Adjusters, Policyholders, Claims Departments, Claim Form Types, Coverage Types, Service Providers
- **6 Batch Facts** — Claims Doc Events, Claim Lifecycle, Adjuster Wellness, Claim Accuracy, Policyholder Satisfaction, Underwriting Docs
- **6 Event Facts** → Eventhouse — Claim Status Changes, Claims System Clicks, Field Inspections, Claim Handoffs, Fraud Alerts, Verification Scans
- **5 Streams** → Eventhouse — Claim Status Changes, Field Adjuster Location, Claims System Clicks, Fraud Alerts, Customer Contacts

### Ontology
- **6 Entity Types:** ClaimsAdjuster, Policyholder, ClaimsDepartment, ClaimFormType, CoverageType, ServiceProvider
- **5 Relationship Types:** assigned_to_dept, handles_claim, uses_form, covers_with, serviced_by_provider
- **6 Contextualizations:** ClaimsDocEvent, ClaimStatusChange, FieldInspection, ClaimHandoff, FraudAlert, VerificationScan

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# INDUSTRY SETTING
# ════════════════════════════════════════════════════════════════════════

INDUSTRY       = "insurance"
INDUSTRY_LABEL = "Insurance"

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# FABRIC ARTIFACT NAMES
# ════════════════════════════════════════════════════════════════════════
# Update these to match your Fabric workspace artifact names.

LAKEHOUSE_NAME      = "InsuranceLakehouse"
WAREHOUSE_NAME      = "Insurance_Datawarehouse"
EVENTHOUSE_NAME     = "insurance_rt_store"
ONTOLOGY_NAME       = "InsuranceDocBurdenOntology"
DATA_AGENT_NAME     = "InsuranceQA"
OPS_AGENT_NAME      = "InsuranceDocBurden"
SEMANTIC_MODEL_NAME = "Insurance_Ops_Model"

print(f"Lakehouse:      {LAKEHOUSE_NAME}")
print(f"Warehouse:      {WAREHOUSE_NAME}")
print(f"Eventhouse:     {EVENTHOUSE_NAME}")
print(f"Ontology:       {ONTOLOGY_NAME}")
print(f"Data Agent:     {DATA_AGENT_NAME}")
print(f"Ops Agent:      {OPS_AGENT_NAME}")
print(f"Semantic Model: {SEMANTIC_MODEL_NAME}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# DATA PATHS & EVENTHOUSE CONNECTION
# ════════════════════════════════════════════════════════════════════════

# CSV files location in Lakehouse Files area
CSV_BASE_PATH = "/lakehouse/default/Files/insurance_claims_operations/data"

# Schemas
LAKEHOUSE_SCHEMA = "dbo"
WAREHOUSE_SCHEMA = "dbo"

# ── Eventhouse Connection ───────────────────────────────────────────────
# Fill these in from your Fabric workspace
EVENTHOUSE_CLUSTER_URI = ""   # e.g. "https://trd-xxxxx.z5.kusto.fabric.microsoft.com"
EVENTHOUSE_DATABASE    = ""   # e.g. "insurance_rt_store"

# Ontology package path
ONTOLOGY_PACKAGE_PATH = "/lakehouse/default/Files/insurance_ops_ontology_package.iq"

print(f"CSV source:     {CSV_BASE_PATH}")
print(f"LH schema:      {LAKEHOUSE_SCHEMA}")
print(f"WH schema:      {WAREHOUSE_SCHEMA}")
print(f"Eventhouse:     {EVENTHOUSE_CLUSTER_URI or '(fill in your cluster URI)'}")
print(f"Ontology pkg:   {ONTOLOGY_PACKAGE_PATH}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# INSURANCE TABLE DEFINITIONS (pre-filled, no auto-discovery needed)
# ════════════════════════════════════════════════════════════════════════

import os, json

# ── 6 Dimension Tables → Lakehouse + Warehouse ──────────────────────────
DIM_TABLES = [
    "dim_adjusters",
    "dim_policyholders",
    "dim_claims_departments",
    "dim_claim_form_types",
    "dim_coverage_types",
    "dim_service_providers",
]

# ── 6 Batch Fact Tables → Lakehouse + Warehouse ─────────────────────────
FACT_TABLES_BATCH = [
    "fact_claims_doc_events",
    "fact_claim_lifecycle",
    "fact_adjuster_wellness",
    "fact_claim_accuracy",
    "fact_policyholder_satisfaction",
    "fact_underwriting_docs",
]

# ── 6 Event Fact Tables → Eventhouse (KQL) ──────────────────────────────
FACT_TABLES_EVENT = [
    "fact_claim_status_changes",
    "fact_claims_system_clicks",
    "fact_field_inspections",
    "fact_claim_handoffs",
    "fact_fraud_alerts",
    "fact_verification_scans",
]

# ── 5 Streaming Tables → Eventhouse (KQL) ───────────────────────────────
STREAM_TABLES = [
    "stream_claim_status_changes",
    "stream_field_adjuster_location",
    "stream_claims_system_clicks",
    "stream_fraud_alerts",
    "stream_customer_contacts",
]

# ── Combined Target Lists ───────────────────────────────────────────────
LAKEHOUSE_TABLES  = DIM_TABLES + FACT_TABLES_BATCH
WAREHOUSE_TABLES  = DIM_TABLES + FACT_TABLES_BATCH + FACT_TABLES_EVENT
EVENTHOUSE_TABLES = FACT_TABLES_EVENT + STREAM_TABLES

# ── KQL Table Name Mapping (CSV name → KQL table name) ─────────────────
# Event facts strip the 'fact_' prefix; streams strip 'stream_' prefix
EVENTHOUSE_KQL_NAMES = {
    "fact_claim_status_changes":     "claim_status_changes",
    "fact_claims_system_clicks":     "claims_system_clicks",
    "fact_field_inspections":        "field_inspections",
    "fact_claim_handoffs":           "claim_handoffs",
    "fact_fraud_alerts":             "fraud_alerts",
    "fact_verification_scans":       "verification_scans",
    "stream_claim_status_changes":   "claim_status_changes_rt",
    "stream_field_adjuster_location":"field_adjuster_location",
    "stream_claims_system_clicks":   "claims_system_clicks_rt",
    "stream_fraud_alerts":           "fraud_alerts_rt",
    "stream_customer_contacts":      "customer_contacts",
}

print(f"{'='*60}")
print(f"INSURANCE TABLE INVENTORY")
print(f"{'='*60}")
print(f"\nDimension tables ({len(DIM_TABLES)}):")
for t in DIM_TABLES: print(f"  • {t}")
print(f"\nFact tables — Batch ({len(FACT_TABLES_BATCH)}):")
for t in FACT_TABLES_BATCH: print(f"  • {t}")
print(f"\nFact tables — Event/Eventhouse ({len(FACT_TABLES_EVENT)}):")
for t in FACT_TABLES_EVENT: print(f"  • {t}  →  {EVENTHOUSE_KQL_NAMES[t]}")
print(f"\nStreaming tables — Eventhouse ({len(STREAM_TABLES)}):")
for t in STREAM_TABLES: print(f"  • {t}  →  {EVENTHOUSE_KQL_NAMES[t]}")
print(f"\n{'─'*60}")
print(f"Lakehouse target:  {len(LAKEHOUSE_TABLES)} tables (12 Delta tables)")
print(f"Warehouse target:  {len(WAREHOUSE_TABLES)} tables (18 SQL tables)")
print(f"Eventhouse target: {len(EVENTHOUSE_TABLES)} tables (11 KQL tables)")
print(f"Total:             23 CSV files")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# EXPECTED ROW COUNTS (for validation in downstream notebooks)
# ════════════════════════════════════════════════════════════════════════

EXPECTED_ROW_COUNTS = {
    # Dimensions (142 total)
    "dim_adjusters":                    30,
    "dim_policyholders":                50,
    "dim_claims_departments":            5,
    "dim_claim_form_types":             22,
    "dim_coverage_types":               15,
    "dim_service_providers":            20,
    # Batch Facts (440 total)
    "fact_claims_doc_events":          200,
    "fact_claim_lifecycle":             60,
    "fact_adjuster_wellness":           30,
    "fact_claim_accuracy":              60,
    "fact_policyholder_satisfaction":    50,
    "fact_underwriting_docs":           40,
    # Event Facts (655 total)
    "fact_claim_status_changes":       180,
    "fact_claims_system_clicks":       300,
    "fact_field_inspections":           45,
    "fact_claim_handoffs":              25,
    "fact_fraud_alerts":                35,
    "fact_verification_scans":          70,
    # Streams (400 total)
    "stream_claim_status_changes":      80,
    "stream_field_adjuster_location":   80,
    "stream_claims_system_clicks":      80,
    "stream_fraud_alerts":              80,
    "stream_customer_contacts":         80,
}

total_lakehouse = sum(EXPECTED_ROW_COUNTS[t] for t in LAKEHOUSE_TABLES)
total_eventhouse = sum(EXPECTED_ROW_COUNTS[t] for t in EVENTHOUSE_TABLES)
print(f"Expected Lakehouse rows:  {total_lakehouse:,}")
print(f"Expected Eventhouse rows: {total_eventhouse:,}")
print(f"Expected total rows:      {sum(EXPECTED_ROW_COUNTS.values()):,}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# SCHEMA INSPECTION HELPER
# ════════════════════════════════════════════════════════════════════════

def preview_table(table_name, base_path=CSV_BASE_PATH, rows=5):
    """Read a CSV and display schema + sample rows."""
    path = f"{base_path}/{table_name}.csv"
    df = spark.read.option("header", True).option("inferSchema", True).csv(path)
    print(f"\n{'─'*60}")
    print(f"Table: {table_name}  |  {df.count()} rows  |  {len(df.columns)} columns")
    print(f"{'─'*60}")
    df.printSchema()
    df.show(rows, truncate=False)
    return df

# Uncomment to preview specific tables:
# preview_table("dim_adjusters")
# preview_table("fact_claims_doc_events")
# preview_table("stream_fraud_alerts")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# CONFIG SUMMARY (exported to downstream notebooks via %run)
# ════════════════════════════════════════════════════════════════════════

CONFIG = {
    "industry":             INDUSTRY,
    "industry_label":       INDUSTRY_LABEL,
    "lakehouse_name":       LAKEHOUSE_NAME,
    "warehouse_name":       WAREHOUSE_NAME,
    "eventhouse_name":      EVENTHOUSE_NAME,
    "eventhouse_uri":       EVENTHOUSE_CLUSTER_URI,
    "eventhouse_db":        EVENTHOUSE_DATABASE,
    "ontology_name":        ONTOLOGY_NAME,
    "ontology_package":     ONTOLOGY_PACKAGE_PATH,
    "data_agent_name":      DATA_AGENT_NAME,
    "ops_agent_name":       OPS_AGENT_NAME,
    "semantic_model_name":  SEMANTIC_MODEL_NAME,
    "csv_base_path":        CSV_BASE_PATH,
    "lakehouse_schema":     LAKEHOUSE_SCHEMA,
    "warehouse_schema":     WAREHOUSE_SCHEMA,
    "dim_tables":           DIM_TABLES,
    "fact_tables_batch":    FACT_TABLES_BATCH,
    "fact_tables_event":    FACT_TABLES_EVENT,
    "stream_tables":        STREAM_TABLES,
    "lakehouse_tables":     LAKEHOUSE_TABLES,
    "warehouse_tables":     WAREHOUSE_TABLES,
    "eventhouse_tables":    EVENTHOUSE_TABLES,
}

print(f"\n{'═'*60}")
print(f"✅ INSURANCE CONFIG READY")
print(f"{'═'*60}")
print(json.dumps({k: v if not isinstance(v, list) else f"{len(v)} tables" for k, v in CONFIG.items()}, indent=2))
print(f"\nThis config is imported by downstream notebooks via:")
print(f"  %run ./Insurance_Config")